# Results: what happened, and why

Pick a run in **Settings**, then **Run All**. Every figure answers one question; the score numbers come from the course grader. Nothing here changes a submission: it only reads `submissions/best.npy`, the run folder and the logs.

If `SAVE_FIGURES` is on, the last cell saves every figure as a PNG in `<run>/figures/` (git-ignored). Copy the ones you want for the report into a tracked `figures/` folder.

In [ ]:
# Setup: work from the repo root; re-read changed linkopt files before each cell.
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

root = Path.cwd()
while not (root / 'linkopt').is_dir() and root != root.parent:
    root = root.parent
os.chdir(root)

import linkopt  # always import first: pins JAX to the CPU
import matplotlib.pyplot as plt

from linkopt import report
from linkopt.submission import load

print('working in', root)

## Settings

- `RUN`: the run folder to analyse (default: the latest).
- `FACTOR`: the setting to compare (default: whichever one this run varied).
- `LEADERBOARD`: hypervolumes copied from the leaderboard (Problem 1, 2, 3), for scale.
- `SAVE_FIGURES`: save every figure as a PNG at the end.

In [ ]:
RUN = report.run_dirs()[-1]          # or e.g. Path('runs/20261002-233526')
FACTOR = None                        # None = whatever this run varied, e.g. 'n_joints'
LEADERBOARD = {                      # from the leaderboard (Problem 1, 2, 3 HV)
    '#1 Shape Grapplers (2026-10-02)': [6.280951, 9.46617, 29.58659],
    '#2 ex-ta (2026-10-02)': [5.514438, 7.914575, 20.837218],
}
SAVE_FIGURES = True

figures = {}  # every figure, by name, for saving at the end
leader = next(iter(LEADERBOARD.values()))
print('run:', RUN)

## 1. How does our submission score?

The grader's numbers for `submissions/best.npy`, the leaderboard's number (the **sum** of the three raw hypervolumes), and how much of each kangaroo's scoring box we cover.

In [ ]:
table = report.score_table()
print(f"{'':32s} {'hypervolume':>11s} {'normalized':>10s} {'box':>6s} {'covered':>8s}")
for r in table['rows']:
    print(f"{r['kangaroo']:32s} {r['hv']:11.3f} {r['hv_norm']:10.3f} {r['box']:6.1f} {r['coverage']:8.1%}")
print(f"\ngrader's overall (average of normalized): {table['overall']:.4f}")
print(f"leaderboard's overall (sum of hypervolumes): {table['leaderboard']:.3f}")
for name, hvs in LEADERBOARD.items():
    print(f'   {name}: {sum(hvs):.3f}  (' + ', '.join(f"{h:.2f}" for h in hvs) + ')')

## 2. Where do our designs sit? (the trade-off)

Each dot is a submitted design; the shaded staircase is the score. **Left:** zoomed to the designs. **Right:** the whole scoring box from utopia (0, 0); the empty area is the score still available. The title compares with the leader.

In [ ]:
scored = report.submission_scores(load(report.BEST_PATH))
for t, (designs, F) in scored.items():
    figures[f'tradeoff_k{t + 1}'] = report.plot_trade_off(F, t, leader_hv=leader[t])
    plt.show()

## 3. What do the best mechanisms look like?

Three designs per kangaroo: the **closest fit**, one from the **middle** of the trade-off, and the one with the **least material**. Left: the mechanism (fixed joints hatched, motor link yellow, the traced joint's path orange). Right: that path over the kangaroo, aligned the way the grader aligns it (shifted and rotated, never resized).

In [ ]:
for t, (designs, F) in scored.items():
    for name, i in report.pick_designs(F).items():
        title = f'{report.KANGAROOS[t]}, {name}'
        figures[f"design_k{t + 1}_{name.replace(' ', '_')}"] = report.plot_design(designs[i], t, title)
        plt.show()

## 4. Every design on the front

Each row: a non-dominated design, its fit, and where it sits on the staircase (red). Up to 6 rows per kangaroo, spread evenly along the front. (The course notebooks' standard figure.)

In [ ]:
for t, (designs, F) in scored.items():
    figures[f'front_k{t + 1}'] = report.plot_front(designs, F, t, max_rows=6)
    plt.show()

## 5. The DOE: what mattered?

Each job's **own** hypervolume (from `jobs.csv`), grouped by the factor. Kangaroos are separate problems: compare levels **within** a row. **The one rule:** an effect is real only if it's bigger than the spread between seeds.

- **Heatmap:** colour = median share of the box; text = median [middle half of the seeds], and how many seeds found nothing.
- **Box plots:** the full spread, one dot per seed.

In [ ]:
rows = report.read_jobs(RUN)
varied = report.factors_varied(rows)
factor = FACTOR or (varied[0] if varied else None)
print(f'{len(rows)} GA jobs; settings varied: {varied or "none"}')
if factor is None:
    print('This run used one value of every setting, so there is nothing to compare.')
else:
    figures['doe_heatmap'] = report.plot_doe_heatmap(rows, factor)
    plt.show()
    figures['doe_boxes'] = report.plot_doe_boxes(rows, factor)
    plt.show()

## 6. Does refinement help?

Each job: hypervolume from the GA alone (x) vs. after gradient refinement (y). Points above the grey diagonal: the local search added something on top of the global search.

In [ ]:
if factor is not None:
    figures['ga_vs_refined'] = report.plot_ga_vs_refined(rows, factor)
    plt.show()

## 7. Do more seeds keep paying?

The hypervolume of all designs from the first *k* seeds, pooled. Where a line flattens, more replicates of that setting stop adding score. (Scores every job's designs once: takes ~10-20 s for a big run.)

In [ ]:
if factor is not None:
    curve = report.seeds_curve(rows, report.job_F(RUN), factor)
    figures['seeds_curve'] = report.plot_seeds_curve(curve, factor)
    plt.show()

## 8. Where did the submission come from?

Which kinds of jobs produced the designs now in `best.npy`, searching every run folder on this computer. "Not in this computer's runs": the starter baseline, or a teammate's merged file.

In [ ]:
prov = report.provenance()
for t, counts in prov.items():
    print(report.KANGAROOS[t], dict(counts.most_common()))
figures['provenance'] = report.plot_provenance(prov)
plt.show()

## Save the figures

In [ ]:
if SAVE_FIGURES:
    for path in report.save_figures(figures, RUN / 'figures'):
        print(path)
else:
    print('SAVE_FIGURES is off: nothing saved')